In [ ]:
print("=" * 80)
print("ROOT CAUSE AND FIX SUMMARY")
print("=" * 80)

summary = """
ROOT CAUSE (THE BUG):
━━━━━━━━━━━━━━━━━━━━
The old main.py preprocessing prepended [1] to every review:
    encoded_review = [1]  ← THIS IS WRONG
    
This [1] is NOT in the training data from imdb.load_data(). It represents
a "start of sequence" metadata marker, not part of actual reviews.

IMPACT:
- Model was trained on sequences like: [w1+3, w2+3, w3+3, ...]
- Inference sent: [1, w1+3, w2+3, w3+3, ...]
- The extra [1] corrupts the embedding layer's input
- All subsequent predictions are based on corrupted semantics
- Sentiment flips to "Positive" for many negative reviews

THE FIX:
━━━━━━━━
Remove the [1] prepend and start with an empty list:
    encoded_review = []  ← START EMPTY
    
This matches the exact format from imdb.load_data():
    Training: Reviews loaded as [w1+3, w2+3, w3+3, ...]
    Inference: Preprocess to [w1+3, w2+3, w3+3, ...]
    Perfect match!

WHAT'S ALREADY DONE:
━━━━━━━━━━━━━━━━━━━
✓ main.py has been updated with the corrected preprocess_text function
✓ The [1] prepend has been removed
✓ All other logic (word indexing, +3 shift, OOV handling) is correct

Next steps:
1. Run your Streamlit app: streamlit run main.py
2. Test with negative reviews - they should now predict "Negative"
3. Test with positive reviews - they should predict "Positive"
"""

print(summary)


## Section 5: Summary and Implementation Guide

In [ ]:
# Test positive reviews to ensure fix works for both classes
positive_reviews = [
    "This movie is amazing! I loved it.",
    "Brilliant and entertaining film.",
    "Outstanding cinematography. Highly recommend!"
]

print("\n" + "=" * 80)
print("TESTING POSITIVE REVIEWS: OLD vs NEW PREPROCESSING")
print("=" * 80)

for review in positive_reviews:
    print(f"\nReview: '{review}'")
    print("-" * 80)
    
    old_input = preprocess_text_OLD(review)
    old_pred = model.predict(old_input, verbose=0)[0][0]
    old_sentiment = "Positive" if old_pred > 0.5 else "Negative"
    old_confidence = old_pred if old_pred > 0.5 else 1 - old_pred
    
    new_input = preprocess_text_NEW(review)
    new_pred = model.predict(new_input, verbose=0)[0][0]
    new_sentiment = "Positive" if new_pred > 0.5 else "Negative"
    new_confidence = new_pred if new_pred > 0.5 else 1 - new_pred
    
    print(f"  OLD (buggy):   {old_sentiment:8s} (confidence: {old_confidence:.4f})")
    print(f"  NEW (correct): {new_sentiment:8s} (confidence: {new_confidence:.4f})")


In [ ]:
# Test cases: clearly negative reviews (label should be 0 - Negative)
test_reviews = [
    "This movie is bad. I hated it.",
    "The story was logicless.",
    "Terrible waste of time. Worst film ever.",
    "Absolutely horrible. Unwatchable garbage."
]

print("=" * 80)
print("TESTING NEGATIVE REVIEWS: OLD vs NEW PREPROCESSING")
print("=" * 80)

for review in test_reviews:
    print(f"\nReview: '{review}'")
    print("-" * 80)
    
    # Old preprocessing
    old_input = preprocess_text_OLD(review)
    old_pred = model.predict(old_input, verbose=0)[0][0]
    old_sentiment = "Positive" if old_pred > 0.5 else "Negative"
    old_confidence = old_pred if old_pred > 0.5 else 1 - old_pred
    
    # New preprocessing
    new_input = preprocess_text_NEW(review)
    new_pred = model.predict(new_input, verbose=0)[0][0]
    new_sentiment = "Positive" if new_pred > 0.5 else "Negative"
    new_confidence = new_pred if new_pred > 0.5 else 1 - new_pred
    
    print(f"  OLD (buggy):   {old_sentiment:8s} (confidence: {old_confidence:.4f})")
    print(f"  NEW (correct): {new_sentiment:8s} (confidence: {new_confidence:.4f})")
    
    if old_sentiment != new_sentiment:
        print(f"  ✓ FIX WORKS! Corrected from {old_sentiment} → {new_sentiment}")


## Section 4: Test Negative Reviews - OLD vs NEW Preprocessing

In [ ]:
print("=" * 70)
print("DEMONSTRATION: HOW THE BUG CORRUPTS INPUT")
print("=" * 70)

test_text = "This movie is bad"
old_encoded = preprocess_text_OLD(test_text)
new_encoded = preprocess_text_NEW(test_text)

print(f"\nOriginal text: {test_text}")
print(f"\nOLD (BUGGY) preprocessing:")
print(f"  First 20 indices: {old_encoded[0][:20]}")
print(f"  Shape: {old_encoded.shape}")
print(f"  Starts with [1]?: {old_encoded[0][0] == 1}")

print(f"\nNEW (CORRECTED) preprocessing:")
print(f"  First 20 indices: {new_encoded[0][:20]}")
print(f"  Shape: {new_encoded.shape}")
print(f"  Starts with padding [0]?: {new_encoded[0][0] == 0}")

print("\n" + "=" * 70)
print("KEY INSIGHT:")
print("=" * 70)
print("The OLD version adds an extra [1] token at the position where")
print("the training data had the first word. This shifts ALL semantic")
print("information in the sequence, confusing the embedding layer.")
print("The LSTM was trained on sequences WITHOUT this [1] prepend.")


In [ ]:
# OLD (BUGGY) PREPROCESSING FUNCTION
def preprocess_text_OLD(text):
    """THE BUGGY VERSION - prepends [1] that wasn't in training data"""
    text = re.sub(r'[^\w\s]', '', text.lower())
    words = text.split()
    
    # BUG: This [1] is NOT in the original imdb.load_data() sequences!
    encoded_review = [1]  # THIS IS THE BUG
    
    for word in words:
        index = word_index.get(word, None)
        if index is None or index >= 10000:
            encoded_review.append(2)
        else:
            encoded_review.append(index + 3)
    
    padded_review = sequence.pad_sequences([encoded_review], maxlen=250)
    return padded_review


# NEW (CORRECTED) PREPROCESSING FUNCTION
def preprocess_text_NEW(text):
    """THE CORRECTED VERSION - matches imdb.load_data() format exactly"""
    text = re.sub(r'[^\w\s]', '', text.lower())
    words = text.split()
    
    # NO [1] prepend - matches training data format exactly
    encoded_review = []  # Start empty like training data
    
    for word in words:
        index = word_index.get(word, None)
        if index is None or index >= 10000:
            encoded_review.append(2)
        else:
            encoded_review.append(index + 3)
    
    padded_review = sequence.pad_sequences([encoded_review], maxlen=250)
    return padded_review


print("✓ Defined OLD (buggy) and NEW (corrected) preprocessing functions")


## Section 3: The Buggy vs Corrected Preprocessing

In [ ]:
# Load the trained LSTM model
model = load_model('lstm_model.h5')

print("=" * 60)
print("TRAINED MODEL ARCHITECTURE")
print("=" * 60)
model.summary()

print("\n" + "=" * 60)
print("MODEL INPUT SPECIFICATIONS")
print("=" * 60)
print(f"Expected input shape: (batch_size, 250)")
print(f"Actual input layer: {model.layers[0]}")
print(f"Input shape details: batch dimension + {max_len} sequence tokens")
print(f"\nEmbedding layer (layer 1):")
print(f"  - Input vocabulary size: {max_features}")
print(f"  - Embedding output dimensions: 128")
print(f"  - Expected input: integer indices from 0 to {max_features-1}")


## Section 2: Examine Model Architecture and Input Requirements

In [ ]:
# Inspect a sample from the training data BEFORE padding
sample_idx = 0
sample_raw = X_train[sample_idx]
sample_label = y_train[sample_idx]

print("=" * 60)
print(f"SAMPLE {sample_idx} FROM TRAINING DATA (BEFORE PADDING)")
print("=" * 60)
print(f"Label: {sample_label} ({'Positive' if sample_label == 1 else 'Negative'})")
print(f"Raw encoded sequence length: {len(sample_raw)}")
print(f"First 20 indices: {sample_raw[:20]}")
print()

# Decode the raw sequence by reversing the +3 shift
decoded_text = ' '.join([reverse_word_index.get(i - 3, '?') for i in sample_raw])
print(f"Decoded text (first 200 chars):\n{decoded_text[:200]}...")


In [ ]:
# Examine the reserved index mapping
print("=" * 60)
print("IMDB RESERVED INDICES:")
print("=" * 60)
print("0: Padding token (used for sequences shorter than max_len)")
print("1: Start of sequence token (metadata, NOT prepended to actual sequences)")
print("2: Out-of-vocabulary (OOV) token for unknown words")
print("3+: Actual word indices (word_index value + 3)")
print("\nExample word mappings:")
print(f"  word_index['movie'] = {word_index.get('movie', 'NOT FOUND')}")
print(f"  Encoded as index: {word_index.get('movie', 0) + 3}")
print(f"  word_index['the'] = {word_index.get('the', 'NOT FOUND')}")
print(f"  Encoded as index: {word_index.get('the', 0) + 3}")


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model
import re

# Load the IMDB dataset as it was loaded during training
max_features = 10000
max_len = 250

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

# Get word index for encoding/decoding
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

print("IMDB Dataset Loaded Successfully")
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
print(f"Word vocabulary size: {len(word_index)}")
print(f"Max sequence length used: {max_len}")


## Section 1: Load and Inspect Training Data Preprocessing

# IMDB Sentiment Analysis: Preprocessing Handshake Fix

## Issue Summary
The model was consistently predicting "Positive" for clearly negative reviews. This notebook diagnoses the root cause (preprocessing handshake error) and validates the fix.

**Root Cause**: The inference script was prepending `[1]` (start token) that wasn't in the training data, corrupting input to the model.

**Solution**: Remove the start token and match the exact encoding format from `imdb.load_data()`.